# 2 — The score

Four components, each scored 0–1, combined by a weighted sum into 0–100.

| Component | Weight | Question it answers |
|---|---|---|
| Industry fit | 35 | Is this the right *sort* of company? |
| Lifecycle fit | 25 | Are they at the stage where this problem bites? |
| Commercial substance | 25 | Can they actually buy anything? |
| Filing health | 15 | Would a salesperson waste a morning here? |

**This is a weighted rank, not a model, and that is a decision rather than a shortcut.** Nobody has
labelled which of these 156,107 companies actually bought software, so there is no target variable to
train against — a fitted model here would be a confident-looking guess carrying a validation score
that measures nothing. A transparent score a sales team will argue with beats an opaque one they
quietly ignore.

In [1]:
import config
import scoring

for component, weight in config.WEIGHTS.items():
    print(f"{component:22} {weight:3d}")
print(f"{'total':22} {sum(config.WEIGHTS.values()):3d}")

industry_fit            35
lifecycle_fit           25
commercial_substance    25
filing_health           15
total                  100


## Industry fit — 35 points

Relevance is **graded, not binary**. A company whose declared main activity is *development of
building projects* is the buyer. A general civil engineering contractor occasionally develops, and is
worth less of a salesperson's morning. Scoring the two identically would rank the list badly while
looking rigorous — the worst failure mode available, because it is invisible.

A code in `SicText_1` is the self-declared main activity. The same code in slots 2–4 is a secondary
line of business, so it takes a 0.8 discount rather than being dropped.

In [2]:
examples = [
    (["41100 - Development of building projects", "", "", ""], "primary, exact buyer"),
    (["56101 - Licensed restaurants", "41100 - Development of building projects", "", ""],
     "same code, secondary activity"),
    (["68100 - Buying and selling of own real estate", "", "", ""], "adjacent"),
    (["42990 - Other civil engineering", "", "", ""], "weak signal"),
    (["62020 - IT consultancy", "", "", ""], "out of segment"),
]
for sics, label in examples:
    print(f"{scoring.industry_fit(sics):.2f}   {label}")

1.00   primary, exact buyer
0.80   same code, secondary activity
0.70   adjacent
0.25   weak signal
0.00   out of segment


## Lifecycle fit — 25 points

A trapezoid, not a threshold. The commercial argument, in order:

- **under 2 years** — a developer with no completed scheme has no budget and no data need
- **2 to 4** — ramping; interest rises as they start bidding on more than one site
- **4 to 9** — the sweet spot: multiple concurrent sites, still on spreadsheets and the planning
  portal, no incumbent tool
- **9 to 20** — decaying; increasingly likely to have bought something already, and to have a
  procurement process that a first call cannot clear

The audit that scoped this project suggested a 2–5 year window borrowed from generic SaaS scale-up
targeting. It is **widened here because development cycles are long**: a housebuilder four years old
may still be on its first scheme. That is exactly the kind of assumption that should change when the
segment changes, which is why it lives in `config.py` as four named numbers.

In [3]:
for age in [1, 2, 3, 4, 6.5, 9, 14.5, 20, 25]:
    bar = "#" * int(scoring.lifecycle_fit(age) * 40)
    print(f"{age:5.1f} yrs   {scoring.lifecycle_fit(age):.2f}  {bar}")

  1.0 yrs   0.00  
  2.0 yrs   0.00  
  3.0 yrs   0.50  ####################
  4.0 yrs   1.00  ########################################
  6.5 yrs   1.00  ########################################
  9.0 yrs   1.00  ########################################
 14.5 yrs   0.50  ####################
 20.0 yrs   0.00  
 25.0 yrs   0.00  


## Commercial substance — 25 points

**This component was added after the first full run, and recording why is more useful than hiding it.**

With only industry, lifecycle and filing health, **4,401 of the 156,107 companies scored exactly
100.0**. A "top 25" drawn from a 4,401-way tie is alphabetical noise wearing the costume of a ranking.
The three original components describe whether a company is the *right sort*. None of them describe
whether it is **big enough to have a budget**.

The free dataset publishes no turnover and no headcount. Two usable proxies remain:

1. **Accounts category.** What a company may file is set by statutory size thresholds, so the category
   is a coarse but genuine size ladder. A developer filing FULL or GROUP accounts is materially larger
   than a micro-entity.
2. **Outstanding mortgage charges, read as a positive signal — which inverts the usual convention.**

   Conventionally a charge is a risk marker: debt secured against company assets, ranking a new
   lender behind existing secured creditors. Credit agencies read a rising charge count as rising
   leverage, and on a credit report five outstanding charges is bad news.

   This is not a credit assessment. The question is *is this worth an hour of a salesperson's time*,
   not *will this company repay a loan*, and charges point opposite ways for those two questions.
   Development finance is how development is funded, so each live charge is roughly a site being
   built right now. **A developer with five live charges is a worse credit risk and a better
   prospect at the same time.** One with none is usually dormant, tiny, or holding land it is not
   developing.

   The cost of that inversion: an over-leveraged developer heading for administration looks
   identical to a busy one on this signal. Filing health catches the ones that stop filing, but a
   company can be distressed and still file on time. This is a prioritisation score, not a
   qualification decision.

   Saturating at three: the difference between one site and three is commercially large, between
   forty and forty-three it is not.

Size carries 65% of the component. A large developer with no live charges is still a buyer; a
micro-entity with three charges is usually a special-purpose vehicle holding a single site.

In [4]:
cases = [
    ("GROUP", 3, "consolidated group, three live sites"),
    ("FULL", 2, "full accounts, two live sites"),
    ("SMALL", 1, "small company, one live site"),
    ("MICRO ENTITY", 3, "micro-entity SPV with finance"),
    ("MICRO ENTITY", 0, "micro-entity shell"),
    ("DORMANT", 0, "dormant"),
]
for category, charges, label in cases:
    print(f"{scoring.commercial_substance(category, charges):.3f}   {label}")

print()
print("Charges saturate at three:",
      scoring.commercial_substance("FULL", 3) == scoring.commercial_substance("FULL", 40))

1.000   consolidated group, three live sites
0.818   full accounts, two live sites
0.442   small company, one live site
0.480   micro-entity SPV with finance
0.130   micro-entity shell
0.000   dormant

Charges saturate at three: True


## Filing health — 15 points

A **disqualifier layer, not a credit assessment**. The only question being asked is whether a
salesperson would waste a morning, and overdue statutory filings are the cheapest proxy for a company
that is dormant, distressed or a shell.

It is deliberately the smallest weight. It works by dragging dead companies down, not by separating
live ones — nearly every active company files on time, so as a *ranking* signal it is nearly constant.

A **missing** due date counts as not overdue. The register leaves the field blank for companies too
newly incorporated to owe a filing, and penalising those would contradict the lifecycle component.

In [5]:
from datetime import date

as_of = date(2026, 9, 1)
rows = [
    (date(2027, 3, 1), "FULL", date(2027, 1, 1), "everything filed on time"),
    (date(2026, 3, 1), "FULL", date(2027, 1, 1), "accounts six months overdue"),
    (date(2027, 3, 1), "DORMANT", date(2027, 1, 1), "dormant but compliant"),
    (None, "FULL", None, "too new to owe a filing"),
    (date(2020, 1, 1), "DORMANT", date(2020, 1, 1), "abandoned shell"),
]
for accounts_due, category, conf_due, label in rows:
    print(f"{scoring.filing_health(accounts_due, category, conf_due, as_of):.2f}   {label}")

1.00   everything filed on time
0.30   accounts six months overdue
0.50   dormant but compliant
1.00   too new to owe a filing
0.00   abandoned shell


## Putting it together

The weights encode two commercial claims worth arguing with:

- **Industry fit is the largest single weight**, because calling the wrong sort of company is the most
  expensive mistake available — it burns the prospect as well as the hour.
- **A perfect-but-irrelevant company caps at 65**, so it can never outrank a good-fit company. That is
  the invariant the 35-point industry weight exists to create, and a test pins it.

In [6]:
print("perfect prospect             ", scoring.total_score(1.0, 1.0, 1.0, 1.0))
print("wrong industry, else perfect ", scoring.total_score(0.0, 1.0, 1.0, 1.0))
print("right industry, too young    ", scoring.total_score(1.0, 0.0, 1.0, 1.0))
print("right industry, no substance ", scoring.total_score(1.0, 1.0, 0.0, 1.0))

perfect prospect              100.0
wrong industry, else perfect  65.0
right industry, too young     75.0
right industry, no substance  75.0


Every rule above is pinned by a test in `test_scoring.py`, each named for the commercial case it
encodes rather than for the arithmetic. Changing a rule therefore means arguing with a sentence, not
just editing a number.

```
python -m pytest -q     # 42 passed
```